In [1]:
# Importações Qiskit 2.3
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import EstimatorV2, SamplerV2
from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms.optimizers import COBYLA
from qiskit_aer import AerSimulator

from typing import List, Tuple, Dict, Optional
from itertools import permutations, combinations

import time
import numpy as np

In [2]:
# Formatação tempo
def format(ns):
  horas = ns // (3600 * 10**9)
  minutos = (ns // (60 * 10**9)) % 60
  segundos = (ns // 10**9) % 60
  milissegundos = (ns // 10**6) % 1000
  nanosegundos = ns % 10**6

  return f"{horas}h {minutos}m {segundos}s {milissegundos}ms {nanosegundos}ns"

In [3]:
_DEGREE_PENALTY_MULTIPLIER: float = 1.5
_SUBTOUR_PENALTY_MULTIPLIER: float = 3.0
_PAULI_COEFF_THRESHOLD: float = 1e-12	

In [4]:
class QAOATSP:    
	def __init__(self, n_vertices: int, distances: np.ndarray, configuration: np.ndarray, seed: int = 42):
		"""
		Args:
			n_vertices: Número de cidades
			distances: Matriz de distâncias (nxn)
			configuration: Configurações para inicialização e mixer
			seed: Seed para reproducibilidade
		"""
		self.n_vertices = n_vertices
		self.distances = distances
		self.configuration = configuration
		self.seed = seed
		np.random.seed(seed)

		# Para TSP: precisamos de uma variável por aresta única (não direcionada)
		# Matriz triangular superior: n(n-1)/2
		self.n_edges = self.n_vertices * (self.n_vertices - 1) // 2
		self.n_qubits = self.n_edges
		
		# Mapear índice de qubit para aresta (i, j)
		self.edge_map = {}
		self.reverse_edge_map = {}
		qubit_idx = 0
		for i in range(self.n_vertices):
			for j in range(i + 1, self.n_vertices):
				self.edge_map[qubit_idx] = (i, j)
				self.reverse_edge_map[(i, j)] = qubit_idx
				self.reverse_edge_map[(j, i)] = qubit_idx
				qubit_idx += 1

		self.simulator = AerSimulator()
		self.estimator = EstimatorV2(mode=self.simulator)
		self.sampler = SamplerV2(mode=self.simulator)

		self.H_cost = self._build_cost_hamiltonian()
		self.H_constraint = self._build_constraint_hamiltonian()
		self.H_subtour = self._build_subtour_hamiltonian()

		self.lp_solution = self._solve_tsp_lp_relaxation()

	def _solve_tsp_lp_relaxation(self) -> np.ndarray:
		from scipy.optimize import linprog
		n = len(self.distances)
		
		edge_index = {}
		edges = []
		idx = 0
		for i in range(n):
			for j in range(i + 1, n):
				edge_index[(i, j)] = idx
				edge_index[(j, i)] = idx  # simétrico
				edges.append((i, j))
				idx += 1
		n_vars = len(edges)  # = n(n-1)/2
		c = np.array([self.distances[i, j] for (i, j) in edges])

		A_eq = np.zeros((n, n_vars))
		b_eq = np.full(n, 2.0)

		for city in range(n):
			for (i, j), var_idx in zip(edges, range(n_vars)):
				if i == city or j == city:
					A_eq[city, var_idx] = 1.0

		bounds = [(0.0, 1.0)] * n_vars
		result = linprog(
			c,
			A_eq=A_eq,
			b_eq=b_eq,
			bounds=bounds,
			method='highs',   # solver mais robusto do scipy
		)
		if not result.success:
			raise ValueError(f"LP não convergiu: {result.message}")

		lp_solution = np.zeros((n, n))
		for (i, j), val in zip(edges, result.x):
			lp_solution[i, j] = val
			lp_solution[j, i] = val  # simétrico

		return lp_solution

	def _build_cost_hamiltonian(self) -> SparsePauliOp:
		pauli_list = []
		for qubit, (i, j) in self.edge_map.items():
			weight = self.distances[i, j]
			pauli_str = ['I'] * self.n_qubits
			pauli_str[qubit] = 'Z'
			pauli_str = ''.join(pauli_str[::-1])
			
			pauli_list.append((pauli_str, -weight / 2.0))

		return SparsePauliOp.from_list(pauli_list).simplify()
	
	def _build_constraint_hamiltonian(self, penalty_weight: float = None) -> SparsePauliOp:
		if penalty_weight is None:
			avg_edge = float(np.mean(self.distances[self.distances > 0]))
			penalty_weight = avg_edge * self.n_vertices * _DEGREE_PENALTY_MULTIPLIER  

		pauli_list = []
		for v in range(self.n_vertices):
			edges_v = [q for q, (i, j) in self.edge_map.items() if i == v or j == v]
			k = len(edges_v)

			# --- Termos lineares (coeficiente correto com sinal) ---
			linear_coeff = penalty_weight * (2.0 - k / 2.0)  

			for q in edges_v:
				pauli_str = ['I'] * self.n_qubits
				pauli_str[q] = 'Z'
				pauli_str = "".join(pauli_str[::-1]) 
				pauli_list.append((pauli_str, linear_coeff))

			# --- Termos quadráticos (interação par a par) ---
			for idx, q1 in enumerate(edges_v):
				for q2 in edges_v[idx + 1:]:
					pauli_str = ['I'] * self.n_qubits
					pauli_str[q1] = 'Z'
					pauli_str[q2] = 'Z'
					pauli_str = "".join(pauli_str[::-1])
					pauli_list.append((pauli_str, penalty_weight / 2.0))

		return SparsePauliOp.from_list(
			[(k, v) for k, v in pauli_list if abs(v) > _PAULI_COEFF_THRESHOLD]
		).simplify()
	

	def _enumerate_hamiltonian_cycles(self, vertices: list) -> list:
		"""
		Retorna todos os ciclos hamiltonianos distintos sobre 'vertices'
		como listas de índices de qubits (arestas do ciclo).
		"""
		k = len(vertices)
		if k < 3:
			return []

		start = vertices[0]
		rest  = vertices[1:]
		seen, cycles = set(), []

		for perm in permutations(rest):
			cycle_v = [start] + list(perm)
			edge_qubits = []
			valid = True

			for i in range(k):
				u, v = cycle_v[i], cycle_v[(i + 1) % k]
				key = (min(u, v), max(u, v))
				if key not in self.reverse_edge_map:
					valid = False
					break
				edge_qubits.append(self.reverse_edge_map[key])

			if not valid:
				continue

			# Deduplicação: ciclo direto e reverso têm o mesmo conjunto de arestas
			canonical = frozenset(edge_qubits)
			if canonical not in seen:
				seen.add(canonical)
				cycles.append(edge_qubits)

		return cycles

	def _add_product_penalty(self, pauli_list: List, qubit_indices: list, weight: float):
		"""
		Adiciona  weight * prod_{q in qubit_indices} x_q  ao pauli_dict.
		x_q = (1 - Z_q) / 2  →  expande via produto de binômios.
		"""
		# Cada chave é um frozenset de qubits que carregam operador Z
		terms = {frozenset(): 1.0}

		for q in qubit_indices:
			new_terms = {}
			for z_set, coeff in terms.items():
				# fator (1/2) * I
				new_terms[z_set] = new_terms.get(z_set, 0.0) + coeff / 2
				# fator (-1/2) * Z_q
				toggled = frozenset(z_set.symmetric_difference({q}))
				new_terms[toggled] = new_terms.get(toggled, 0.0) - coeff / 2
			terms = new_terms

		for z_set, coeff in terms.items():
			if abs(coeff) < _PAULI_COEFF_THRESHOLD:
				continue
			pauli_str = ['I'] * self.n_qubits
			for q in z_set:
				pauli_str[q] = 'Z'
			pauli_str = ''.join(pauli_str[::-1])          
			pauli_list.append((pauli_str, weight * coeff))

	def _build_subtour_hamiltonian(self, penalty_weight: float = None) -> SparsePauliOp:
		"""
		Restrição de eliminação de subciclos (SEC).

		Para cada subconjunto S ⊂ V com 3 ≤ |S| ≤ n-1, enumera todos os
		ciclos hamiltonianos sobre S e adiciona penalidade proporcional ao
		produto das variáveis de aresta: λ · ∏_{e ∈ C} x_e.

		Com as restrições de grau já garantindo ∑_e x_e = 2 em cada vértice,
		isso é suficiente para eliminar todos os subciclos.
		"""

		if penalty_weight is None:
			avg_edge = float(np.mean(self.distances[self.distances > 0]))
			penalty_weight = avg_edge * self.n_vertices * _SUBTOUR_PENALTY_MULTIPLIER

		pauli_list = []

		for k in range(3, self.n_vertices):          # tamanho do subconjunto
			for subset in combinations(range(self.n_vertices), k):
				for cycle_edges in self._enumerate_hamiltonian_cycles(list(subset)):
					self._add_product_penalty(pauli_list, cycle_edges, penalty_weight)

		if not pauli_list:
			# Retorna operador nulo compatível
			return SparsePauliOp.from_list([('I' * self.n_qubits, 0.0)])

		return SparsePauliOp.from_list(
			[(k, v) for k, v in pauli_list if abs(v) > _PAULI_COEFF_THRESHOLD]
		).simplify()

	def _get_total_hamiltonian(self) -> SparsePauliOp:
		return (self.H_cost + self.H_constraint + self.H_subtour).simplify()
	  
	def _build_xy_mixer_hamiltonian(self) -> SparsePauliOp:
		pauli_list = []
		for q1 in range(self.n_qubits):
			for q2 in range(q1 + 1, self.n_qubits):
				edge1 = self.edge_map[q1]
				edge2 = self.edge_map[q2]
				
				if any(v in edge2 for v in edge1):
					x_str = list('I' * self.n_qubits)
					x_str[q1] = 'X'
					x_str[q2] = 'X'
					pauli_list.append(("".join(x_str[::-1]), 0.5))
					
					y_str = list('I' * self.n_qubits)
					y_str[q1] = 'Y'
					y_str[q2] = 'Y'
					pauli_list.append(("".join(y_str[::-1]), 0.5))

		return SparsePauliOp.from_list(pauli_list).simplify()  
	  
	def _build_mixer_hamiltonian(self) -> SparsePauliOp:
		pauli_list = []
		for i in range(self.n_qubits):
			pauli_str = ['I'] * self.n_qubits
			pauli_str[i] = 'X'
			pauli_list.append(("".join(pauli_str[::-1]), 1.0))

		return SparsePauliOp.from_list(pauli_list).simplify()	  

	def _build_mixer_warm_start(self):
		pauli_list = []
		for qubit, (i, j) in self.edge_map.items():
			x_e = float(self.lp_solution[i, j])
			x_coeff = -np.sqrt(x_e)
			z_coeff = np.sqrt(1.0 - x_e)
			if abs(x_coeff) > 1e-10:
				x_label = ['I'] * self.n_qubits
				x_label[qubit] = 'X'
				pauli_list.append(("".join(x_label[::-1]), x_coeff))
			if abs(z_coeff) > 1e-10:
				z_label = ['I'] * self.n_qubits
				z_label[qubit] = 'Z'
				pauli_list.append(("".join(z_label[::-1]), z_coeff))

		return SparsePauliOp.from_list(pauli_list).simplify()

	def _apply_warm_start(self, qc: QuantumCircuit):
		for qubit, (i, j) in self.edge_map.items():
			x_relaxed = self.lp_solution[i, j]          
			theta = 2 * np.arcsin(np.sqrt(x_relaxed))
			qc.ry(theta, qubit)

	def _apply_dicke_state(self, circuit: QuantumCircuit, bias: float = 0.8) -> None:
		def _tour_distance(tour: List[int]) -> float:
			return sum(self.distances[tour[i], tour[(i + 1) % len(tour)]] for i in range(len(tour)))
					
		def nearest_neighbour_tour(start: int = 0) -> List[int]:
			n = self.n_vertices
			visited = {start}
			tour = [start]
			current = start
			while len(visited) < n:
				unvisited = [c for c in range(n) if c not in visited]
				nearest = min(unvisited, key=lambda c: self.distances[current, c])
				tour.append(nearest)
				visited.add(nearest)
				current = nearest
			return tour

		def two_opt_improve(tour: List[int]) -> List[int]:
			best = tour[:]
			n = len(best)
			improved = True
			while improved:
				improved = False
				for i in range(1, n - 1):
					for j in range(i + 1, n):
						candidate = best[:i] + best[i:j+1][::-1] + best[j+1:]
						if _tour_distance(candidate) < _tour_distance(best):
							best = candidate
							improved = True
			return best

		def multi_start_nearest_neighbour(n_starts: Optional[int] = None) -> List[int]:
			n = self.n_vertices
			starts = range(n) if n_starts is None else range(min(n_starts, n))
			best_tour: Optional[List[int]] = None
			best_dist = float("inf")

			for start in starts:
				tour = nearest_neighbour_tour(start=start)
				tour = two_opt_improve(tour)
				dist = _tour_distance(tour)
				if dist < best_dist:
					best_dist = dist
					best_tour = tour

			return best_tour

		def tour_to_qubit_indices(tour: List[int], edge_to_qubit: dict) -> List[int]:
			n = len(tour)
			qubit_indices = []
			for i in range(n):
				city_a = tour[i]
				city_b = tour[(i + 1) % n]
				qubit_indices.append(edge_to_qubit[(city_a, city_b)])
			return qubit_indices

		n_cities = self.n_vertices
		k = n_cities 

		best_tour = multi_start_nearest_neighbour()
		tour_qubits = tour_to_qubit_indices(best_tour, self.reverse_edge_map)

		high_angle = np.pi * bias         # θ for edges IN the tour   → amplitude ≈ bias
		low_angle  = np.pi * (1 - bias)   # θ for edges NOT in tour   → amplitude ≈ 1−bias

		tour_set = set(tour_qubits)
		for qubit in range(self.n_qubits):
			angle = high_angle if qubit in tour_set else low_angle
			circuit.ry(angle, qubit)

	def _create_qaoa_circuit(self, params: np.ndarray) -> QuantumCircuit:
		p = len(params) // 2
		beta_params = params[:p]
		gamma_params = params[p:]
		
		qc = QuantumCircuit(self.n_qubits)

		if self.configuration['initialize'] == 'hadamard':
			qc.h(range(self.n_qubits))
		elif self.configuration['initialize'] == 'warm_start':
			self._apply_warm_start(qc)
		elif self.configuration['initialize'] == 'dicke':
			self._apply_dicke_state(qc)
		else:
			print('Estado inicial não definido')
			return
		
		for layer in range(p):
			self._apply_hamiltonian(qc, self._get_total_hamiltonian(), gamma_params[layer])
			
			mixer = None
			if self.configuration['mixer'] == 'X':
				mixer = self._build_mixer_hamiltonian()
			elif self.configuration['mixer'] == 'XY':
				mixer = self._build_xy_mixer_hamiltonian()
			elif self.configuration['mixer'] == 'Y':
				mixer = self._build_mixer_warm_start()
			else:			 
				print('Mixer não definido')
				return
			self._apply_hamiltonian(qc, mixer, beta_params[layer])

		return qc
	
	def _apply_hamiltonian(self, qc: QuantumCircuit, 
						  hamiltonian: SparsePauliOp, 
						  time: float) -> None:
		for pauli_str, coeff in hamiltonian.to_list():
			coeff_real = float(np.real(coeff))
			if abs(coeff_real) < 1e-10:
				continue

			pauli_rev = pauli_str[::-1]  # pauli_rev[i] = operador no qubit i
			active = [(i, p) for i, p in enumerate(pauli_rev) if p != 'I']

			if not active:
				continue  # fase global — ignorar

			indices = [i for i, _ in active]
			ops     = {i: p for i, p in active}

			# 1. Rotação de base: X→Z (via H), Y→Z (via S†H)
			for i, p in active:
				if p == 'X':
					qc.h(i)
				elif p == 'Y':
					qc.sdg(i)
					qc.h(i)
				# Z: não precisa de rotação

			# 2. Cadeia de CNOTs para computar a paridade
			for k in range(len(indices) - 1):
				qc.cx(indices[k], indices[k + 1])

			# 3. Rotação RZ no último qubit (único ponto com ângulo)
			qc.rz(2.0 * coeff_real * time, indices[-1])

			# 4. Desfaz a cadeia de CNOTs (ordem inversa)
			for k in range(len(indices) - 2, -1, -1):
				qc.cx(indices[k], indices[k + 1])

			# 5. Desfaz rotação de base
			for i, p in active:
				if p == 'X':
					qc.h(i)
				elif p == 'Y':
					qc.h(i)
					qc.s(i)

	def optimize(self, p: int = 1, max_iter: int = 100) -> Dict:
		best_result = None
		best_energy = float('inf')

		gamma_init = np.random.uniform(0, np.pi / 2, p)
		beta_init  = np.random.uniform(0, np.pi,     p)
		initial_params = np.concatenate([beta_init, gamma_init])

		energy_history = []
		def objective_function(params):
			try:
				qc = self._create_qaoa_circuit(params)
				job = self.estimator.run([(qc, self._get_total_hamiltonian())])
				energy = float(job.result()[0].data.evs)
				energy_history.append(energy)
				return energy
			except:
				return 1e10

		optimizer = COBYLA(maxiter=max_iter, tol=1e-4)
		result = optimizer.minimize(objective_function, initial_params)

		if result.fun < best_energy:
			best_energy = result.fun
			best_result = {
				'optimal_params': result.x,
				'optimal_energy': result.fun,
				'energy_history': energy_history,
				'num_params': 2 * p,
			}

		return best_result
	
	def _two_opt(self, tour: List[int]) -> List[int]:
		best = tour[:]
		n = len(best)
		improved = True
		while improved:
			improved = False
			for i in range(1, n - 1):
				for j in range(i + 1, n):
					new_tour = best[:i] + best[i:j+1][::-1] + best[j+1:]
					if self._calculate_tour_distance(new_tour) < self._calculate_tour_distance(best):
						best = new_tour
						improved = True
		return best
	
	def get_solution(self, params: np.ndarray, n_shots: int = 1024) -> Tuple[List, float, Dict]:
		qc = self._create_qaoa_circuit(params)
		qc.measure_all()

		result = self.sampler.run([qc], shots=n_shots).result()
		counts = result[0].data.meas.get_counts()
		
		unique_tours = {}
		for bitstring, count in counts.items():
			tour = self._bitstring_to_tour(bitstring)
			if self.configuration['two_opt']:
				tour = self._two_opt(tour)   
			norm_tour = self._normalize_tour(tour)
			tour_key = tuple(norm_tour)
			
			if tour_key not in unique_tours:
				unique_tours[tour_key] = {
					'count': count, 
					'distance': self._calculate_tour_distance(norm_tour)
				}
			else:
				unique_tours[tour_key]['count'] += count
				
		sorted_by_distance = sorted(unique_tours.items(), key=lambda x: x[1]['distance'])
		best_tour, best_data = sorted_by_distance[0]
		
		stats = {
			'probability': best_data['count'] / n_shots,
			'top_by_distance': sorted_by_distance
		}
		return list(best_tour), best_data['distance'], stats

	def _normalize_tour(self, tour: List[int]) -> List[int]:
		"""Normaliza o tour para que sempre comece pelo menor índice e tenha direção única."""
		if not tour: return []
		idx = tour.index(min(tour))
		shifted = tour[idx:] + tour[:idx]
		if len(shifted) > 2 and shifted[1] > shifted[-1]:
			shifted = [shifted[0]] + shifted[:0:-1]
		return shifted

	def _bitstring_to_tour(self, bitstring: str) -> List[int]:
		bitstring = bitstring[::-1]
		edges = [self.edge_map[i] for i, bit in enumerate(bitstring) if bit == '1']
		
		adj = {i: [] for i in range(self.n_vertices)}
		for u, v in edges:
			adj[u].append(v)
			adj[v].append(u)
			
		tour = [0]
		visited = {0}
		curr = 0
		while len(visited) < self.n_vertices:
			options = [v for v in adj[curr] if v not in visited]
			if not options: # Fallback guloso se o grafo for desconexo
				remaining = [v for v in range(self.n_vertices) if v not in visited]
				next_v = min(remaining, key=lambda x: self.distances[curr, x])
			else:
				next_v = options[0]
			tour.append(next_v)
			visited.add(next_v)
			curr = next_v
		return tour

	def _calculate_tour_distance(self, tour):
		return sum(self.distances[tour[i], tour[(i+1)%len(tour)]] for i in range(len(tour)))
	
	def solve_classical(self) -> Tuple[List, float]:
		"""Solução clássica por força bruta (para comparação)"""
		start = time.perf_counter_ns()
		best_tour = None
		best_distance = float('inf')
		
		for perm in permutations(range(1, self.n_vertices)):
			tour = [0] + list(perm)
			distance = self._calculate_tour_distance(tour)
			if distance < best_distance:
				best_distance = distance
				best_tour = tour

		return best_tour, best_distance, time.perf_counter_ns() - start 

In [ ]:
results_summary = []

M3 = np.array([[ 0.0, 6.5, 11.0 ], 
			   [ 6.5, 0.0, 11.5 ], 
			   [ 11.0, 11.5, 0.0 ] ])

M4 = np.array([[ 0.0,  14.5, 10.0,  9.5 ], 
			   [ 14.5,  0.0, 15.5,  9.5 ], 
			   [ 10.0, 15.5,  0.0,  2.5 ], 
			   [  9.5,  9.5,  2.5,  0.0 ]])

M5 = np.array([[  0.0,  11.0,  10.0,  3.0,  10.5 ], 
 			   [ 11.0,   0.0,  10.0,   4.0,  13.0 ],
 			   [ 10.0,  10.0,   0.0,  13.0,   7.5 ],
 			   [  3.0,   4.0,  13.0,   0.0,  16.0 ],
 			   [ 10.5,  13.0,   7.5,  16.0,   0.0 ]]) 

M6 = np.array([[ 0.0,  14.0,  13.5,  4.0,  15.5, 11.5 ],
 			   [ 14.0,  0.0,  14.5,  1.5,  8.5,  8.0 ],
 			   [ 13.5, 14.5,  0.0,  10.5, 12.0,  5.5 ],
			   [ 4.0,   1.5, 10.5,  0.0,  12.0, 7.5],
			   [ 15.5,  8.5, 12.0,  12.0,   0.0,  16.0 ],
			   [ 11.5,  8.0,  5.5, 7.5, 16.0, 0.0 ]])

experiments = {
	3: { 'p_layer':  3, 'distance': M3 },
  	4: { 'p_layer':  4, 'distance': M4 },
	5: { 'p_layer':  6, 'distance': M5 },
	6: { 'p_layer':  10, 'distance': M6 }
}

configurations = [
	{ 'initialize': 'hadamard', 'mixer': 'X', 'two_opt': False },
	{ 'initialize': 'warm_start', 'mixer': 'Y', 'two_opt': False },
	{ 'initialize': 'warm_start', 'mixer': 'X', 'two_opt': False },
	{ 'initialize': 'warm_start', 'mixer': 'XY', 'two_opt': False },
	{ 'initialize': 'warm_start', 'mixer': 'XY', 'two_opt': True },
	{ 'initialize': 'dicke', 'mixer': 'X', 'two_opt': False },
	{ 'initialize': 'dicke', 'mixer': 'XY', 'two_opt': False }
]

n_shots = 1024
max_iter = 400

for configuration in configurations:
	for n_vertices, data in experiments.items():
		p_layer = data['p_layer']
		distance = data['distance']
		initialize = configuration['initialize']
		mixer = configuration['mixer']
		two_opt = configuration['two_opt']

		opt = QAOATSP(n_vertices=n_vertices, distances=distance, configuration=configuration)

		start = time.perf_counter_ns()
		opt_result = opt.optimize(p=p_layer, max_iter=max_iter)
		tour, _, stats = opt.get_solution(opt_result['optimal_params'], n_shots=n_shots)

		# Obter solução
		tour, distance, stats = opt.get_solution(
			opt_result['optimal_params'], 
			n_shots=n_shots * 4
		)
		finished = time.perf_counter_ns() - start 

		print(f"{n_vertices} VÉRTICES initialize: {initialize} mixer: {mixer} two_opt: {two_opt}")
		print("-" * 70)
		print('******* QAOA ******')
		print(f'Shots {n_shots} rotas: {len(stats["top_by_distance"])} COBYLA max_iter: {max_iter} qubits usados: {opt.n_qubits} camadas: {p_layer} parâmetros B/G: {opt_result['num_params']}')
		print(f"Rota: {tour} menor distância: {distance:.2f} tempo processamento: {format(finished)} ns: {finished}")

		classical_tour, classical_distance, classical_finished = opt.solve_classical()
		print('****** CLÁSSICO ******')
		print(f"Rota: {classical_tour} menor distância: {classical_distance:.2f} tempo processamento: {format(classical_finished)} ns: {classical_finished}")

		print('****** QAOA/CLÁSSICO ******')
		print(f"Taxa aproximação: {distance / classical_distance:.4f} Prob. melhor solução: {stats['probability']:.4f}")

		print("\nRotas únicas:")
		for i, (t, data) in enumerate(stats['top_by_distance'], 1):
			print(f"{i}. {list(t)} - distance: {data['distance']:.2f} counts: {data['count']}")        

		results_summary.append({
			'initialize': initialize,
			'mixer': mixer,
			'two_opt': two_opt,
			'vertex': n_vertices,
			'p_layer': p_layer,
			'distance': distance,
			'route': tour,
			'time': format(finished),
			'probability': stats['probability'],
			'classical_route': classical_tour,
			'classical_distance': classical_distance,
			'classiscal_time': format(classical_finished)
		})
		print("\n")


# Cabeçalho manual com nomes abreviados para caber na tela
header = f"{'Init':<10} | {'Mix':<8} | {'2_opt':<5} | {'Ver':<4} | {'P':<2} | {'Dist Q':<10} | {'Prob.':<10} | {'Time Q':<24} | {'Dist C':<10} | {'Time C':<20}"
print(header)
print("-" * len(header))

for res in results_summary:
    # Usando f-strings diretamente para cada valor
    print(f"{str(res['initialize']):<10} | "
          f"{str(res['mixer']):<8} | "
          f"{str(res['two_opt']):<5} | "
          f"{res['vertex']:<4} | "
          f"{res['p_layer']:<2} | "
          f"{res['distance']:<10.4f} | "
		  f"{res['probability']:<10.4f} | "
          f"{res['time']:<8} | "
          f"{res['classical_distance']:<10.4f} | "
          f"{res['classiscal_time']:<8}")

3 VÉRTICES initialize: hadamard mixer: X two_opt: False
----------------------------------------------------------------------
******* QAOA ******
Shots 1024 rotas: 1 COBYLA max_iter: 400 qubits usados: 3 camadas: 3 parâmetros B/G: 6
Rota: [0, 1, 2] menor distância: 29.00 tempo processamento: 0h 0m 2s 436ms 714500ns ns: 2436714500
****** CLÁSSICO ******
Rota: [0, 1, 2] menor distância: 29.00 tempo processamento: 0h 0m 0s 0ms 21300ns ns: 21300
****** QAOA/CLÁSSICO ******
Taxa aproximação: 1.0000 Prob. melhor solução: 1.0000

Rotas únicas:
1. [0, 1, 2] - distance: 29.00 counts: 4096


4 VÉRTICES initialize: hadamard mixer: X two_opt: False
----------------------------------------------------------------------
******* QAOA ******
Shots 1024 rotas: 3 COBYLA max_iter: 400 qubits usados: 6 camadas: 4 parâmetros B/G: 8
Rota: [0, 1, 3, 2] menor distância: 36.50 tempo processamento: 0h 0m 5s 98ms 739400ns ns: 5098739400
****** CLÁSSICO ******
Rota: [0, 1, 3, 2] menor distância: 36.50 tempo proc